In [1]:
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu

# Load dataset (first 100 for demo)
dataset = load_dataset('opus100', 'en-fr', split='train[:100]')

# Load tokenizer and model
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Prepare English sentences
english_sentences = [ex['translation']['en'] for ex in dataset]
reference_translations = [[ex['translation']['fr']] for ex in dataset]

# Tokenize input
inputs = tokenizer(english_sentences, return_tensors="pt", padding=True, truncation=True)

# Generate translations
translated_tokens = model.generate(**inputs)
french_translations = [tokenizer.decode(t, skip_special_tokens=True) for t in translated_tokens]

# Print sample translations
for en, fr in zip(english_sentences[:5], french_translations[:5]):
    print(f"EN: {en}")
    print(f"FR: {fr}")
    print("-" * 40)

# Compute BLEU score
bleu = sacrebleu.corpus_bleu(french_translations, reference_translations)
print("BLEU score:", bleu.score)

C:\Users\23adsb61\.conda\envs\pytorch61\lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

C:\Users\23adsb61\.conda\envs\pytorch61\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\23adsb61\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-fr. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


EN: The time now is 05:08 .
FR: Le temps est maintenant 05:08 .
----------------------------------------
EN: This Regulation shall enter into force on the seventh day following its publication in the Official Journal of the European Union.
FR: Le présent règlement entre en vigueur le septième jour suivant celui de sa publication au Journal officiel de l'Union européenne.
----------------------------------------
EN: Hello, what's that?
FR: Bonjour, c'est quoi ?
----------------------------------------
EN: And then I will teach you everything i know.
FR: Et je t'apprendrai tout ce que je sais.
----------------------------------------
EN: Did you find something?
FR: Vous avez trouvé quelque chose ?
----------------------------------------
BLEU score: 14.535768424205482


In [2]:
# ======================================
# Interactive English → French Translator with BLEU
# ======================================

import warnings
import os
import torch
from transformers import MarianMTModel, MarianTokenizer
from datasets import load_dataset
import sacrebleu

# --------------------------
# Suppress warnings
# --------------------------
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

# --------------------------
# Load MarianMT model
# --------------------------
model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("🇬🇧 English → 🇫🇷 French Translator (type 'exit' to quit)\n")

# --------------------------
# Load OPUS100 dataset
# --------------------------
dataset = load_dataset("opus100", "en-fr")
test_data = dataset["test"]

print("Dataset loaded successfully.")
print("Sample entry:", dataset["train"][0])

# --------------------------
# Translation function
# --------------------------
def translate_batch(text_list, num_beams=5, max_length=128):

    inputs = tokenizer(
        text_list,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    translated_tokens = model.generate(
        **inputs,
        num_beams=num_beams,
        max_length=max_length
    )

    outputs = [
        tokenizer.decode(t, skip_special_tokens=True)
        .replace("▁", " ")
        .strip()
        for t in translated_tokens
    ]

    return outputs


# --------------------------
# Example translation
# --------------------------
example_text = "Artificial intelligence is changing the world"

example_translation = translate_batch([example_text])[0]

print("\nExample Translation")
print("English:", example_text)
print("French :", example_translation)


# --------------------------
# BLEU evaluation on 500 sentences
# --------------------------
print("\nCalculating BLEU score on first 500 test sentences...")

batch_size = 32
predictions = []
references = []

for i in range(0, 500, batch_size):

    batch_en = [
        test_data[j]["translation"]["en"]
        for j in range(i, min(i + batch_size, 500))
    ]

    batch_fr = [
        [test_data[j]["translation"]["fr"]]
        for j in range(i, min(i + batch_size, 500))
    ]

    batch_pred = translate_batch(batch_en)

    predictions.extend(batch_pred)
    references.extend(batch_fr)


# Clean text
predictions_clean = [
    pred.replace("▁", " ").strip()
    for pred in predictions
]

references_clean = [
    [ref[0].replace("▁", " ").strip()]
    for ref in references
]

# BLEU score
bleu = sacrebleu.corpus_bleu(predictions_clean, references_clean)

print("BLEU Score on 500 sentences:", bleu.score)


# --------------------------
# Multiple sentence example
# --------------------------
sentences = [
    "Machine learning is powerful",
    "Natural language processing is interesting",
    "Deep learning models are widely used"
]

print("\nMultiple Sentence Translation Example:")

for sentence in sentences:

    translated = translate_batch([sentence])[0]

    print("\nEnglish:", sentence)
    print("French :", translated)


# --------------------------
# Interactive translation
# --------------------------
while True:

    user_input = input("\nEnter English sentence: ")

    if user_input.lower() == "exit":
        print("Exiting translator...")
        break

    french_output = translate_batch([user_input])[0]

    print("French translation:", french_output)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🇬🇧 English → 🇫🇷 French Translator (type 'exit' to quit)

Dataset loaded successfully.
Sample entry: {'translation': {'en': 'The time now is 05:08 .', 'fr': 'The time now is 05:05 .'}}

Example Translation
English: Artificial intelligence is changing the world
French : L'intelligence artificielle change le monde

Calculating BLEU score on first 500 test sentences...
BLEU Score on 500 sentences: 14.751171815982019

Multiple Sentence Translation Example:

English: Machine learning is powerful
French : L'apprentissage automatique est puissant

English: Natural language processing is interesting
French : Le traitement du langage naturel est intéressant

English: Deep learning models are widely used
French : Les modèles d'apprentissage profond sont largement utilisés



Enter English sentence:  hi how are you. i am fine


French translation: Salut, comment allez-vous. Je vais bien.



Enter English sentence:  exit


Exiting translator...
